# GCC Chennai PGR Complaints — Severity classifier (Sentence-Transformers + CatBoost)

**Data:** GCC ERP PGR complaints FY 2024-25 (sheet *PGR Data for FY 2024-25*), cleaned, de-duplicated and labelled by **severity — Low / Medium / High / Critical** — from the complaint description.

| Severity | Meaning |
|---|---|
| Low | requests, administrative or cosmetic issues (new facilities, certificates/tax, parking/encroachment, park upkeep) |
| Medium | routine service failures (street light out, garbage pick-up, stray dogs, potholes, debris) |
| High | public-health hazard or serious disruption (water stagnation, sewage overflow, mosquito/dengue, long-pending/repeat complaints, risk to children/elderly) |
| Critical | immediate danger to life/safety (electric shock, live/fallen wires, dog bites, trees about to fall, open manholes, accidents, sewage in drinking water) |

**Features:** 384-d multilingual sentence embedding of the description **+ the portal complaint type** as a native CatBoost categorical feature.

**Pipeline**
1. Load the labelled dataset (`pgr_labelled_for_model.zip`)
2. EDA — class balance, text length, language mix
3. Leakage control — de-duplicate identical texts before splitting (thousands of complaints share boilerplate text)
4. Stratified train / validation / test split (70 / 15 / 15)
5. Sentence embeddings with a **multilingual** Sentence-Transformer (handles English, Tamil and Tanglish)
6. CatBoost multiclass model (embeddings + categorical complaint type) with early stopping — train/validation **loss curves**, accuracy and F1 curves per iteration
7. Full evaluation on the held-out test set — accuracy, balanced accuracy, precision/recall/F1 (macro/weighted/micro), MCC, Cohen's κ, log-loss, top-k accuracy, ordinal metrics (quadratic-weighted κ, MAE in levels, under-triage rate), ROC-AUC & PR-AUC (one-vs-rest), confusion matrices, ROC/PR curves, calibration, confidence analysis, error analysis, learning curve, baselines
8. Save model + inference helper

> Runtime: **GPU (T4)** recommended — *Runtime → Change runtime type → T4 GPU*. On a T4 the full run takes roughly 15–25 minutes.

In [ ]:
!pip -q install sentence-transformers catboost

In [ ]:
import os, re, json, time, zipfile, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    f1_score, matthews_corrcoef, cohen_kappa_score, log_loss, top_k_accuracy_score, roc_auc_score,
    average_precision_score, classification_report, confusion_matrix, roc_curve, precision_recall_curve, auc)
from catboost import CatBoostClassifier, Pool
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')
OUT = 'outputs'; os.makedirs(OUT, exist_ok=True)

## 1. Load the labelled dataset

In [ ]:
DATA_ZIP, DATA_CSV = 'pgr_labelled_for_model.zip', 'pgr_labelled_for_model.csv'
if not (os.path.exists(DATA_ZIP) or os.path.exists(DATA_CSV)):
    try:
        from google.colab import files
        print('Upload pgr_labelled_for_model.zip (or the .csv)')
        files.upload()
    except ImportError:
        raise FileNotFoundError('Put pgr_labelled_for_model.zip next to this notebook')
if not os.path.exists(DATA_CSV):
    zipfile.ZipFile(DATA_ZIP).extractall('.')
df = pd.read_csv(DATA_CSV)
print(df.shape)
df.head()

## 2. Exploratory analysis

In [ ]:
LABELS = ['Low', 'Medium', 'High', 'Critical']
assert (df.SEVERITY_ID.map(dict(enumerate(LABELS))) == df.SEVERITY).all()
N_CLASSES = len(LABELS)
SHORT = [l if len(l) <= 28 else l[:26] + '…' for l in LABELS]

SEV_COLORS = ['#4C9F70', '#E1B530', '#E07B39', '#C0392B']
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
vc = df.SEVERITY.value_counts().reindex(LABELS)
ax[0].bar(vc.index, vc.values, color=SEV_COLORS)
for i, v in enumerate(vc.values): ax[0].text(i, v, f'{v:,}\n({v/len(df):.1%})', ha='center', va='bottom', fontsize=9)
ax[0].set_title('Complaints per severity'); ax[0].set_ylabel('count'); ax[0].set_ylim(0, vc.max() * 1.18)
df['n_words'] = df.TEXT_FOR_MODEL.str.split().str.len()
sns.histplot(df.n_words.clip(upper=100), bins=50, ax=ax[1]); ax[1].set_title('Words per description (clipped at 100)')
df.LANGUAGE.value_counts().plot.pie(autopct='%1.1f%%', ax=ax[2]); ax[2].set_ylabel(''); ax[2].set_title('Script')
plt.tight_layout(); plt.savefig(f'{OUT}/eda.png'); plt.show()
xt = pd.crosstab(df.CATEGORY, df.SEVERITY)[LABELS]
plt.figure(figsize=(9, 6)); sns.heatmap(xt.div(xt.sum(1), axis=0), annot=True, fmt='.2f', cmap='OrRd')
plt.title('Severity mix within each complaint category'); plt.ylabel(''); plt.tight_layout(); plt.savefig(f'{OUT}/category_severity.png'); plt.show()

## 3. Leakage control & split
Many complaints are call-centre boilerplate (e.g. *"so many dogs are around above mention area…"* appears >11k times).
If identical texts land in both train and test, the test score is inflated. We collapse to **unique normalised texts**
per complaint type (label = majority severity of that text) and split those.

In [ ]:
df['NORM'] = (df.TEXT_FOR_MODEL.str.lower().str.replace(r'[^\w\s]', ' ', regex=True)
              .str.replace(r'\s+', ' ', regex=True).str.strip())
u = (df.groupby(['NORM', 'COMPLAINTTYPENAME']).agg(text=('TEXT_FOR_MODEL', 'first'),
                            label_id=('SEVERITY_ID', lambda s: s.value_counts().index[0]),
                            freq=('SEVERITY_ID', 'size')).reset_index().rename(columns={'COMPLAINTTYPENAME': 'ctype'}))
print(f'{len(df):,} complaints -> {len(u):,} unique texts')

train_df, tmp = train_test_split(u, test_size=0.30, stratify=u.label_id, random_state=SEED)
val_df, test_df = train_test_split(tmp, test_size=0.50, stratify=tmp.label_id, random_state=SEED)
for n, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'{n:5s} {len(d):7,}')
assert not (set(zip(train_df.NORM, train_df.ctype)) & set(zip(test_df.NORM, test_df.ctype)))
y_tr, y_va, y_te = train_df.label_id.values, val_df.label_id.values, test_df.label_id.values

dist = pd.DataFrame({n: d.label_id.value_counts(normalize=True).sort_index() for n, d in
                     [('train', train_df), ('val', val_df), ('test', test_df)]})
dist.index = LABELS; dist.round(4)

## 4. Sentence embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
EMB_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'   # 50+ languages incl. Tamil; 384-d
encoder = SentenceTransformer(EMB_MODEL, device=DEVICE)
encoder.max_seq_length = 128

def embed(texts, name):
    path = f'{OUT}/emb_{name}.npy'
    if os.path.exists(path):
        return np.load(path)
    t0 = time.time()
    E = encoder.encode(list(texts), batch_size=256 if DEVICE == 'cuda' else 64, show_progress_bar=True,
                       convert_to_numpy=True, normalize_embeddings=True)
    print(f'{name}: {E.shape} in {time.time() - t0:.1f}s')
    np.save(path, E); return E

X_tr, X_va, X_te = embed(train_df.text, 'train'), embed(val_df.text, 'val'), embed(test_df.text, 'test')
print('Embedding dim:', X_tr.shape[1])

# CatBoost inputs: embedding columns + complaint type as a native categorical feature
EMB_COLS = [f'e{i}' for i in range(X_tr.shape[1])]
def to_frame(E, d):
    F = pd.DataFrame(E, columns=EMB_COLS); F['ctype'] = d.ctype.astype(str).values; return F
F_tr, F_va, F_te = to_frame(X_tr, train_df), to_frame(X_va, val_df), to_frame(X_te, test_df)
CAT = ['ctype']

### Embedding space (t-SNE of a 4,000-text sample)

In [ ]:
from sklearn.manifold import TSNE
idx = np.random.RandomState(SEED).choice(len(X_tr), min(4000, len(X_tr)), replace=False)
Z = TSNE(n_components=2, init='pca', perplexity=35, random_state=SEED).fit_transform(X_tr[idx])
plt.figure(figsize=(11, 8))
for k in range(N_CLASSES):
    m = y_tr[idx] == k
    plt.scatter(Z[m, 0], Z[m, 1], s=6, color=SEV_COLORS[k], label=LABELS[k], alpha=.7)
plt.legend(markerscale=3, fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.title('t-SNE of sentence embeddings (train sample)'); plt.tight_layout(); plt.savefig(f'{OUT}/tsne.png'); plt.show()

## 5. Train CatBoost (with per-iteration train/validation curves)

In [ ]:
params = dict(
    loss_function='MultiClass',
    eval_metric='TotalF1:average=Macro',
    custom_metric=['MultiClass', 'Accuracy', 'TotalF1:average=Macro', 'TotalF1:average=Weighted'],
    iterations=3000, learning_rate=0.08, depth=6, l2_leaf_reg=3,
    auto_class_weights='SqrtBalanced',
    od_type='Iter', od_wait=150, use_best_model=True,
    random_seed=SEED, verbose=200,
    task_type='GPU' if DEVICE == 'cuda' else 'CPU',
)
if params['task_type'] == 'CPU':
    params['thread_count'] = -1
model = CatBoostClassifier(**params)
t0 = time.time()
model.fit(Pool(F_tr, y_tr, cat_features=CAT), eval_set=Pool(F_va, y_va, cat_features=CAT))
TRAIN_TIME = time.time() - t0
print(f'Training time: {TRAIN_TIME:.1f}s | best iteration: {model.get_best_iteration()} | trees: {model.tree_count_}')

### Loss & metric curves

In [ ]:
ev = model.get_evals_result()
tr_key, va_key = 'learn', 'validation'
def pick(d, metric):
    # CatBoost names logged metrics e.g. 'Accuracy:use_weights=false'; prefer the unweighted series
    for k in (f'{metric}:use_weights=false', f'{metric};use_weights=false', metric):
        if k in d: return np.array(d[k])
    ks = [k for k in d if k.split(':')[0] == metric.split(':')[0] and metric.split(':')[-1].split(';')[0] in k]
    return np.array(d[ks[0]]) if ks else np.array([])
def curve(metric):
    return pick(ev[tr_key], metric), pick(ev[va_key], metric)

metrics_to_plot = [('MultiClass', 'Multiclass log-loss'), ('Accuracy', 'Accuracy'),
                   ('TotalF1:average=Macro', 'Macro F1'), ('TotalF1:average=Weighted', 'Weighted F1')]
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
best = model.get_best_iteration()
for ax, (m, title) in zip(axes.ravel(), metrics_to_plot):
    tr, va = curve(m)
    if len(tr): ax.plot(tr, label='train', lw=1.6)
    if len(va): ax.plot(va, label='validation', lw=1.6)
    ax.axvline(best, ls='--', c='grey', lw=1, label=f'best iter {best}')
    ax.set_title(title); ax.set_xlabel('iteration'); ax.legend()
plt.suptitle('CatBoost training curves', fontsize=14); plt.tight_layout()
plt.savefig(f'{OUT}/training_curves.png'); plt.show()

tr_l, va_l = curve('MultiClass')
gap = pd.DataFrame({'iteration': [best], 'train_logloss': [tr_l[best] if len(tr_l) > best else np.nan],
                    'val_logloss': [va_l[best] if len(va_l) > best else np.nan]})
gap['generalisation_gap'] = gap.val_logloss - gap.train_logloss
gap

## 6. Evaluation on the held-out test set

In [ ]:
t0 = time.time(); P_te = model.predict_proba(F_te); INF_TIME = time.time() - t0
pred = P_te.argmax(1)
P_tr = model.predict_proba(F_tr); P_va = model.predict_proba(F_va)
Y_te_bin = label_binarize(y_te, classes=range(N_CLASSES))

def scores(y, P):
    p = P.argmax(1)
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, average='macro', zero_division=0)
    pw, rw, fw, _ = precision_recall_fscore_support(y, p, average='weighted', zero_division=0)
    Yb = label_binarize(y, classes=range(N_CLASSES))
    return {'accuracy': accuracy_score(y, p), 'balanced_accuracy': balanced_accuracy_score(y, p),
            'precision_macro': pr, 'recall_macro': rc, 'f1_macro': f1,
            'precision_weighted': pw, 'recall_weighted': rw, 'f1_weighted': fw,
            'f1_micro': f1_score(y, p, average='micro'),
            'mcc': matthews_corrcoef(y, p), 'cohen_kappa': cohen_kappa_score(y, p),
            'log_loss': log_loss(y, P, labels=range(N_CLASSES)),
            'top2_accuracy': top_k_accuracy_score(y, P, k=2, labels=range(N_CLASSES)),
            'top3_accuracy': top_k_accuracy_score(y, P, k=3, labels=range(N_CLASSES)),
            'roc_auc_ovr_macro': roc_auc_score(Yb, P, average='macro'),
            'roc_auc_ovr_weighted': roc_auc_score(Yb, P, average='weighted'),
            'pr_auc_macro': average_precision_score(Yb, P, average='macro'),
            # ordinal / triage metrics (severity is ordered Low < Medium < High < Critical)
            'quadratic_weighted_kappa': cohen_kappa_score(y, p, weights='quadratic'),
            'mae_levels': np.abs(y - p).mean(),
            'under_triage_rate': (p < y).mean(),
            'over_triage_rate': (p > y).mean(),
            'critical_recall': ((p == 3) & (y == 3)).sum() / max((y == 3).sum(), 1),
            'critical_precision': ((p == 3) & (y == 3)).sum() / max((p == 3).sum(), 1),
            'high_or_critical_recall': ((p >= 2) & (y >= 2)).sum() / max((y >= 2).sum(), 1)}

summary = pd.DataFrame({'train': scores(y_tr, P_tr), 'validation': scores(y_va, P_va), 'test': scores(y_te, P_te)})
summary.loc['train_time_s'] = [TRAIN_TIME, np.nan, np.nan]
summary.loc['inference_ms_per_text'] = [np.nan, np.nan, 1000 * INF_TIME / len(X_te)]
summary.round(4)

### Per-class report

In [ ]:
rep = pd.DataFrame(classification_report(y_te, pred, target_names=LABELS, output_dict=True, zero_division=0)).T
per_auc = [roc_auc_score(Y_te_bin[:, k], P_te[:, k]) for k in range(N_CLASSES)]
per_ap = [average_precision_score(Y_te_bin[:, k], P_te[:, k]) for k in range(N_CLASSES)]
rep.loc[LABELS, 'roc_auc'] = per_auc; rep.loc[LABELS, 'pr_auc'] = per_ap
rep.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
r = rep.loc[LABELS, ['precision', 'recall', 'f1-score']].copy(); r.index = SHORT
r.sort_values('f1-score').plot.barh(ax=ax, width=.8)
ax.set_xlim(0, 1); ax.set_title('Per-class precision / recall / F1 (test)'); plt.tight_layout()
plt.savefig(f'{OUT}/per_class_f1.png'); plt.show()

### Confusion matrices

In [ ]:
cm = confusion_matrix(y_te, pred)
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(1, 2, figsize=(16, 6.5))
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', xticklabels=SHORT, yticklabels=SHORT, ax=ax[0], cbar=False)
ax[0].set_title('Confusion matrix (counts)')
sns.heatmap(cmn, annot=True, fmt='.2f', cmap='Blues', xticklabels=SHORT, yticklabels=SHORT, ax=ax[1], vmin=0, vmax=1)
ax[1].set_title('Confusion matrix (row-normalised = recall)')
for a in ax: a.set_xlabel('predicted'); a.set_ylabel('true')
plt.tight_layout(); plt.savefig(f'{OUT}/confusion_matrix.png'); plt.show()

off = pd.DataFrame([(LABELS[i], LABELS[j], cm[i, j]) for i in range(N_CLASSES) for j in range(N_CLASSES) if i != j],
                   columns=['true', 'predicted', 'count']).sort_values('count', ascending=False).head(10)
off

### ROC and Precision-Recall curves (one-vs-rest)

In [ ]:
pal = SEV_COLORS
fig, ax = plt.subplots(1, 2, figsize=(16, 6.5))
for k in range(N_CLASSES):
    fpr, tpr, _ = roc_curve(Y_te_bin[:, k], P_te[:, k])
    ax[0].plot(fpr, tpr, color=pal[k], lw=1.3, label=f'{SHORT[k]} ({auc(fpr, tpr):.3f})')
    p, r, _ = precision_recall_curve(Y_te_bin[:, k], P_te[:, k])
    ax[1].plot(r, p, color=pal[k], lw=1.3, label=f'{SHORT[k]} ({per_ap[k]:.3f})')
ax[0].plot([0, 1], [0, 1], 'k--', lw=.8); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR')
ax[0].set_title(f"ROC (macro AUC {summary.loc['roc_auc_ovr_macro', 'test']:.3f})")
ax[1].set_xlabel('recall'); ax[1].set_ylabel('precision'); ax[1].set_title(f"Precision-Recall (macro AP {summary.loc['pr_auc_macro', 'test']:.3f})")
for a in ax: a.legend(fontsize=7.5, loc='lower right' if a is ax[0] else 'lower left')
plt.tight_layout(); plt.savefig(f'{OUT}/roc_pr_curves.png'); plt.show()

### Calibration & confidence

In [ ]:
conf = P_te.max(1); correct = pred == y_te
fig, ax = plt.subplots(1, 3, figsize=(19, 5))
frac, mean_p = calibration_curve(correct, conf, n_bins=15, strategy='quantile')
ax[0].plot(mean_p, frac, 'o-', label='CatBoost'); ax[0].plot([0, 1], [0, 1], 'k--', lw=.8, label='perfect')
ax[0].set_xlabel('predicted confidence'); ax[0].set_ylabel('observed accuracy'); ax[0].set_title('Reliability diagram (top-1)'); ax[0].legend()
bins = np.linspace(0, 1, 16); ids = np.digitize(conf, bins) - 1
ece = sum(abs(correct[ids == b].mean() - conf[ids == b].mean()) * (ids == b).mean() for b in range(15) if (ids == b).any())
ax[0].text(.05, .9, f'ECE = {ece:.4f}', transform=ax[0].transAxes)
sns.histplot(x=conf, hue=np.where(correct, 'correct', 'wrong'), bins=40, ax=ax[1], stat='density', common_norm=False)
ax[1].set_title('Confidence of correct vs wrong predictions')
th = np.linspace(0, .95, 40)
cov = [(conf >= t).mean() for t in th]; acc_t = [correct[conf >= t].mean() for t in th]
ax[2].plot(th, cov, label='coverage'); ax[2].plot(th, acc_t, label='accuracy on covered')
ax[2].set_xlabel('confidence threshold'); ax[2].set_title('Accuracy vs coverage (auto-routing trade-off)'); ax[2].legend()
plt.tight_layout(); plt.savefig(f'{OUT}/calibration_confidence.png'); plt.show()
print(f'ECE: {ece:.4f}')

### Error analysis — most confident mistakes

In [ ]:
err = test_df.assign(direction=np.where(pred < y_te, 'under-triaged', 'over-triaged'), true=[LABELS[i] for i in y_te], predicted=[LABELS[i] for i in pred], confidence=conf)[~correct]
pd.set_option('display.max_colwidth', 140)
err.sort_values('confidence', ascending=False)[['text', 'ctype', 'true', 'predicted', 'direction', 'confidence']].head(20)

## 7. Weighted evaluation on all test complaints
The test set holds unique texts; this re-weights by how many raw complaints each text represents, i.e. the accuracy the department would see in production.

In [ ]:
w = test_df.freq.values
print(f'Complaint-weighted accuracy: {np.average(correct, weights=w):.4f}')
print(f'Complaint-weighted macro F1:  {f1_score(y_te, pred, average="macro", sample_weight=w):.4f}')

## 8. Baselines for context
Majority class, logistic regression on embeddings only, CatBoost on embeddings only (no complaint type), and the full model.

In [ ]:
base = {}
dummy = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)
base['Majority class'] = dummy.predict(X_te)
t0 = time.time()
lr = LogisticRegression(max_iter=2000, C=4, class_weight='balanced', n_jobs=-1).fit(X_tr, y_tr)
print(f'LogReg train {time.time() - t0:.1f}s')
base['LogReg (embeddings)'] = lr.predict(X_te)
ab = {k: v for k, v in params.items() if k != 'verbose'}; ab.update(verbose=0)
cb_emb = CatBoostClassifier(**ab).fit(X_tr, y_tr, eval_set=(X_va, y_va))
base['CatBoost (embeddings only)'] = cb_emb.predict(X_te).ravel().astype(int)
base['CatBoost (embeddings + type)'] = pred
bl = pd.DataFrame({k: {'accuracy': accuracy_score(y_te, v), 'f1_macro': f1_score(y_te, v, average='macro'),
                       'f1_weighted': f1_score(y_te, v, average='weighted'), 'mcc': matthews_corrcoef(y_te, v),
                       'qw_kappa': cohen_kappa_score(y_te, v, weights='quadratic'), 'critical_recall': ((v == 3) & (y_te == 3)).sum() / (y_te == 3).sum()}
                   for k, v in base.items()}).T
bl.round(4)

## 9. Learning curve (training-set size vs performance)

In [ ]:
fracs = [0.05, 0.1, 0.25, 0.5, 1.0]
lc = []
lc_params = {k: v for k, v in params.items() if k not in ('verbose',)}
lc_params.update(iterations=800, verbose=0)
for f in fracs:
    if f < 1:
        sub, _ = train_test_split(np.arange(len(X_tr)), train_size=f, stratify=y_tr, random_state=SEED)
    else:
        sub = np.arange(len(X_tr))
    m = CatBoostClassifier(**lc_params).fit(Pool(F_tr.iloc[sub], y_tr[sub], cat_features=CAT), eval_set=Pool(F_va, y_va, cat_features=CAT))
    lc.append({'train_size': len(sub),
               'train_f1': f1_score(y_tr[sub], m.predict(F_tr.iloc[sub]).ravel().astype(int), average='macro'),
               'val_f1': f1_score(y_va, m.predict(F_va).ravel().astype(int), average='macro')})
lc = pd.DataFrame(lc)
plt.figure(figsize=(8, 5))
plt.plot(lc.train_size, lc.train_f1, 'o-', label='train macro-F1'); plt.plot(lc.train_size, lc.val_f1, 'o-', label='validation macro-F1')
plt.xscale('log'); plt.xlabel('training texts (log)'); plt.ylabel('macro F1'); plt.title('Learning curve'); plt.legend()
plt.tight_layout(); plt.savefig(f'{OUT}/learning_curve.png'); plt.show()
lc.round(4)

## 10. Feature importance

In [ ]:
fi = model.get_feature_importance(type='PredictionValuesChange'); names = np.array(model.feature_names_)
print(f"Complaint type importance: {fi[names == 'ctype'][0]:.2f}%  |  all embedding dims: {fi[names != 'ctype'].sum():.2f}%")
top = np.argsort(fi)[::-1][:25]
plt.figure(figsize=(12, 4)); plt.bar(names[top], fi[top], color=['#C0392B' if n == 'ctype' else '#4C72B0' for n in names[top]]); plt.xticks(rotation=70)
plt.title('Top-25 features by CatBoost importance (red = complaint type)'); plt.tight_layout(); plt.savefig(f'{OUT}/feature_importance.png'); plt.show()

## 11. Save artefacts & inference

In [ ]:
model.save_model(f'{OUT}/catboost_pgr.cbm')
json.dump({'labels': LABELS, 'embedding_model': EMB_MODEL}, open(f'{OUT}/label_map.json', 'w'), indent=1)
metrics = {'test': summary['test'].dropna().to_dict(), 'validation': summary['validation'].dropna().to_dict(),
           'train': summary['train'].dropna().to_dict(), 'ece': float(ece), 'best_iteration': int(best),
           'train_time_s': TRAIN_TIME, 'n_train': len(X_tr), 'n_val': len(X_va), 'n_test': len(X_te)}
json.dump(metrics, open(f'{OUT}/metrics.json', 'w'), indent=1, default=float)
rep.to_csv(f'{OUT}/classification_report.csv'); summary.to_csv(f'{OUT}/metrics_summary.csv')
pd.DataFrame(ev[tr_key]).add_prefix('train_').join(pd.DataFrame(ev[va_key]).add_prefix('val_')).to_csv(f'{OUT}/training_history.csv', index_label='iteration')

def classify(texts, ctypes, k=2):
    E = encoder.encode(list(texts), normalize_embeddings=True)
    P = model.predict_proba(to_frame(E, pd.DataFrame({'ctype': ctypes})))
    return [[(LABELS[i], round(float(p[i]), 3)) for i in np.argsort(p)[::-1][:k]] for p in P]

demo = [('Live electric wire hanging low near the bus stop, children pass daily', 'Fallen EB cables'),
        ('Street dog bit two people in our street yesterday', 'Street Dogs'),
        ('Garbage not collected for 4 days, very bad smell', 'Removal of Garbage'),
        ('Rain water stagnating on our street after yesterday rain', 'Stagnation of Water'),
        ('தெருவில் குப்பை அகற்றப்படவில்லை', 'Removal of Garbage'),
        ('Street light not burning in our street', 'Non burning of Street lights'),
        ('Please provide a new street light at the corner', 'New Street lights'),
        ('Need birth certificate correction', 'Issue of Birth and Death Certificate')]
for (t, c), r in zip(demo, classify([d[0] for d in demo], [d[1] for d in demo])): print(f'{t[:62]:62s} -> {r}')

In [ ]:
# Download all outputs (Colab)
!zip -qr pgr_model_outputs.zip outputs -x 'outputs/emb_*'
try:
    from google.colab import files; files.download('pgr_model_outputs.zip')
except ImportError:
    pass